# 19 — 图自监督与预训练：遮蔽、对比学习与 GraphMAE

监督标签昂贵，但节点特征和图结构常常大量存在。本课学习如何先构造不依赖人工标签的预训练目标，再把编码器迁移到少标签节点分类。核心问题不是‘预训练能否把 loss 降下来’，而是：**在相同下游模型、标签量和训练预算下，它是否稳定超过从头训练？**

## 1. 学习目标

完成本课后，你应该能解释节点/边遮蔽、两视图对比学习、GraphMAE 的 encoder–decoder 结构、linear probing 与 fine-tuning 的区别，以及为什么预训练数据范围和增强策略也是归纳偏置。实验保留 MLP 与同架构 scratch GCN，使用 validation 选型、test 最后查看，并报告多 seed 波动。

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys, time
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.citation import load_planetoid
from crosscity.models.graph_pretraining import GraphEncoder, GraphMAE, PretrainedNodeClassifier
from crosscity.models.static_gnn import MLPNodeClassifier
from crosscity.training.graph_pretraining import (clone_encoder, few_label_training_data,
    pretrain_contrastive, pretrain_graphmae)
from crosscity.training.node_classification import train_node_classifier
from crosscity.utils import seed_everything
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch:', torch.__version__)


## 2. Mac 轻验证与服务器正式模式

`RUN_FULL=False` 只减少预训练轮数、下游轮数和随机种子，不改变 Cora、划分或评价指标。远程服务器设为 `True` 后从头运行。Cora 的 full-batch 实验不要求 neighbor-sampling 后端。

In [ ]:
RUN_FULL = False
SEEDS = [42, 43, 44] if RUN_FULL else [42]
PRETRAIN_EPOCHS = 300 if RUN_FULL else 20
DOWNSTREAM_EPOCHS = 300 if RUN_FULL else 40
PATIENCE = 50 if RUN_FULL else 15
HIDDEN = 256 if RUN_FULL else 64
LABELS_PER_CLASS = 10 if RUN_FULL else 3
data = load_planetoid(ROOT / 'data/raw/planetoid', 'cora')
few_data = few_label_training_data(data, LABELS_PER_CLASS, seed=2026)
pd.Series({'nodes': data.num_nodes, 'features': data.num_features, 'classes': data.num_classes,
           'original train labels': int(data.train_mask.sum()),
           'few-shot train labels': int(few_data.train_mask.sum()),
           'validation': int(data.val_mask.sum()), 'test': int(data.test_mask.sum())})


## 3. 三类自监督目标

**节点遮蔽**用 mask token 替换一部分 $x_v$，再从邻域重建原特征：$\mathcal L_{MAE}=|M|^{-1}\sum_{v\in M}\|g(f(X_{masked},A))_v-x_v\|^2$。loss 只在被遮蔽节点上计算，否则模型可复制未遮蔽输入。**边遮蔽**隐藏部分边并预测端点是否相连，需要严格负采样；第 10 课已覆盖这一路径。**对比学习**对同一图产生两个增强视图，使同一节点表示接近、其他节点作为负例：$\ell_i=-\log\frac{e^{sim(z_i,z'_i)/\tau}}{\sum_j e^{sim(z_i,z'_j)/\tau}}$。

增强不是无害的数据处理：删边可能破坏桥，feature dropout 可能删除类别关键字段。应记录增强强度并做敏感性分析。GraphMAE 不需要大量负例，通常比全节点 InfoNCE 更省显存。

## 4. 泄漏边界

本课是 **transductive self-supervision**：预训练可读取整张 Cora 的无标签特征和边，包括 validation/test 节点，但绝不读取它们的标签。这适合部署时目标图已经存在的场景。如果目标节点或未来边在部署时尚未出现，这种协议会高估 inductive 能力；此时应只在训练子图预训练，再在未见图上评价。少标签函数只缩小 train mask，validation/test 完全不变。

In [ ]:
assert torch.equal(few_data.val_mask, data.val_mask)
assert torch.equal(few_data.test_mask, data.test_mask)
assert not torch.any(few_data.train_mask & few_data.val_mask)
print(torch.bincount(few_data.y[few_data.train_mask], minlength=data.num_classes))


## 5. Smoke run

先验证 mask、重建形状、梯度和下游接口。这里的 loss 数值不能用于结论。

In [ ]:
smoke_data = few_data.to(DEVICE)
seed_everything(0)
smoke_mae = GraphMAE(data.num_features, 16, layers=2, dropout=0).to(DEVICE)
smoke_history = pretrain_graphmae(smoke_mae, smoke_data, epochs=2, mask_ratio=0.4)
smoke_classifier = PretrainedNodeClassifier(smoke_mae.encoder, 16, data.num_classes).to(DEVICE)
smoke_result = train_node_classifier(smoke_classifier, smoke_data, max_epochs=2, patience=2)
print('pretrain losses:', [round(row['loss'], 5) for row in smoke_history],
      '| logits:', tuple(smoke_classifier(smoke_data.x, smoke_data.edge_index).shape))


## 6. 预训练 GraphMAE 与对比编码器

两种方法使用相同两层 GCN 编码器和 hidden width。对比学习只抽样至多 512 个 anchor 构造相似度矩阵，以避免 $O(N^2)$ 显存；消息传递仍读取完整图。预训练 loss 只能用于检查优化是否正常，不能在 GraphMAE 与 InfoNCE 之间横向比较，因为尺度和目标不同。

In [ ]:
pretrained = {}
pretrain_histories = {}
full_data = data.to(DEVICE)
for seed in SEEDS:
    seed_everything(seed)
    mae = GraphMAE(data.num_features, HIDDEN, layers=2, dropout=0.2).to(DEVICE)
    pretrain_histories[(seed, 'GraphMAE')] = pretrain_graphmae(
        mae, full_data, epochs=PRETRAIN_EPOCHS, mask_ratio=0.4)
    pretrained[(seed, 'GraphMAE')] = deepcopy(mae.encoder)

    seed_everything(seed)
    contrastive = GraphEncoder(data.num_features, HIDDEN, layers=2, dropout=0.2).to(DEVICE)
    pretrain_histories[(seed, 'Contrastive')] = pretrain_contrastive(
        contrastive, full_data, epochs=PRETRAIN_EPOCHS, feature_drop=0.2, edge_drop=0.2,
        sample_size=min(512, data.num_nodes))
    pretrained[(seed, 'Contrastive')] = deepcopy(contrastive)


In [ ]:
for (seed, method), history in pretrain_histories.items():
    if seed == SEEDS[0]:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.loss, label=method)
plt.xlabel('pretraining epoch'); plt.ylabel('objective (not cross-method comparable)')
plt.yscale('log'); plt.legend(); plt.show()


## 7. 公平下游比较

`Scratch-GCN` 与预训练方法拥有完全相同的 encoder。`MAE-linear` 冻结 encoder，只训练线性头，测量表示本身是否线性可用；`MAE-finetune` 和 `Contrastive-finetune` 更新全部参数。MLP 是无图少标签基线。每个下游运行都深拷贝 encoder，避免一次微调污染下一次实验。

In [ ]:
records = []
for seed in SEEDS:
    variants = {
        'MLP': MLPNodeClassifier(data.num_features, HIDDEN, data.num_classes),
        'Scratch-GCN': PretrainedNodeClassifier(
            GraphEncoder(data.num_features, HIDDEN, layers=2), HIDDEN, data.num_classes),
        'MAE-linear': PretrainedNodeClassifier(
            clone_encoder(pretrained[(seed, 'GraphMAE')]), HIDDEN, data.num_classes),
        'MAE-finetune': PretrainedNodeClassifier(
            clone_encoder(pretrained[(seed, 'GraphMAE')]), HIDDEN, data.num_classes),
        'Contrastive-finetune': PretrainedNodeClassifier(
            clone_encoder(pretrained[(seed, 'Contrastive')]), HIDDEN, data.num_classes),
    }
    for parameter in variants['MAE-linear'].encoder.parameters(): parameter.requires_grad_(False)
    for name, model in variants.items():
        seed_everything(seed)
        model = model.to(DEVICE)
        started = time.perf_counter()
        result = train_node_classifier(model, few_data.to(DEVICE), max_epochs=DOWNSTREAM_EPOCHS,
            patience=PATIENCE, learning_rate=0.01, weight_decay=5e-4)
        records.append({'seed': seed, 'method': name, 'best_epoch': result.best_epoch,
            'validation_accuracy': result.validation_accuracy, 'test_accuracy': result.test_accuracy,
            'seconds': time.perf_counter() - started})
results = pd.DataFrame(records)
results.groupby('method')[['validation_accuracy', 'seconds']].agg(['mean', 'std'])


## 8. 冻结方案后查看 test

先按 validation 写下选择，再显示下表。只有预训练方法在多个 seed 下稳定超过同架构 `Scratch-GCN`，才能称为预训练收益；超过 MLP 只说明图可能有用。还应把预训练时间计入总成本，不能只报告下游秒数。

In [ ]:
results.groupby('method')[['validation_accuracy','test_accuracy','seconds']].agg(['mean','std'])


## 9. 失败结果同样重要

预训练不提升可能因为重建了与标签无关的高频词、增强破坏语义、Cora 太小、预训练和下游任务不对齐，或 scratch 已足够。linear probe 差而 fine-tune 好，说明表示需要适配；两者都不如 scratch，则预训练目标可能带来负迁移。不能通过给预训练方法更多 hidden、更多下游 epoch 或更多标签来制造优势。

## 10. 课后练习

1. 扫描 mask ratio 0.1/0.4/0.7，同时报告重建 loss 与下游结果。
2. 分别只做 edge dropout、只做 feature dropout，检查增强敏感性。
3. 比较每类 1/3/10/20 个标签并画 learning curve；预训练收益通常应在少标签区更明显。
4. 改为 inductive 协议：预训练时删除 validation/test 节点相关边。
5. 实现 scaled cosine reconstruction loss，解释稀疏词袋上 MSE 的局限。
6. 统计预训练+下游总时间、峰值显存和均值±标准差。

下一课会进一步追问：预训练编码器何时只是‘一个数据集上的初始化’，何时才有资格讨论跨图迁移、graph prompt 和图基础模型。